In [ ]:
!pip install import-ipynb

In [ ]:
import sys
import os
from google.colab import drive
drive.mount('DRIVE_PATH')
os.chdir('CLIP_MODEL_PATH')

import import_ipynb
from contrastive_loss import ContrastiveLoss
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_clip(train_loader, val_loader, model, epochs=10, lr=1e-4, checkpoint=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    loss_fn = ContrastiveLoss()
    loss_trn = []
    loss_val = []
    # Recommend AdamW for Transformer/CLIP models
    # AdamW applies weight decay unlike Adam
    # weight_decay = Adjusts its internal weights so that the model will not inflate its weights to massive number
    # weight_new = weight_old - (Learning Rate x Gradient) - (decay * weight_old)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # INITIALIZE THE SCHEDULER
    # T_max is the number of epochs. It tells the scheduler exactly how long
    # it has to smoothly reduce the learning rate down to near-zero
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    if checkpoint is not None:
      print("Restoring state")
      optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
      scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
      start_epoch = checkpoint['epoch'] + 1
      best_val_loss = checkpoint['best_val_loss']
      loss_trn = checkpoint.get('loss_trn', [])
      loss_val = checkpoint.get('loss_val', [])
      count = checkpoint.get('count', 0)
    else:
      start_epoch = 0
      best_val_loss = 1000

    best_model_state = model.state_dict()
    count = 0
    for i in range(start_epoch, epochs):
        model.train()
        total_loss_trn = 0
        for img, txt, mask in train_loader:
            img = img.to(device)
            txt = txt.to(device)
            mask = mask.to(device) if mask is not None else None
            optimizer.zero_grad()
            img_features, txt_features = model(img, txt, mask)
            loss = loss_fn(img_features, txt_features)
            total_loss_trn += loss.item()
            loss.backward()
            optimizer.step()

        loss_trn.append(total_loss_trn / len(train_loader))
        model.eval()
        with torch.no_grad():
            total_loss_val = 0
            for img, txt, mask in val_loader:
                img = img.to(device)
                txt = txt.to(device)
                mask = mask.to(device) if mask is not None else None
                img_features, txt_features = model(img, txt, mask)
                loss = loss_fn(img_features, txt_features)
                total_loss_val += loss.item()
        loss_val.append(total_loss_val / len(val_loader))

        if loss_val[-1] < best_val_loss:
            torch.save(model.state_dict(), '/content/drive/MyDrive/Colab Notebooks/CLIP Model/best_model_cc3m.pth')
            best_val_loss = loss_val[-1]
            best_model_state = model.state_dict()
            count = 0
        else:
          count += 1

        torch.save({
            'epoch': i,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'loss_trn': loss_trn,
            'loss_val': loss_val,
            'count': count,
        }, f"/content/drive/MyDrive/Colab Notebooks/CLIP Model/checkpoint_epoch_{i+1}.pth")

        print(f"Epoch {i+1}/{epochs}, Training Loss: {loss_trn[-1]:.4f}, Validation Loss: {loss_val[-1]:.4f}")
        if count >= 5:
          print(f"Early stopping at epoch {i+1}, best val loss: {best_val_loss:.4f}")
          model.load_state_dict(best_model_state)
          return model, loss_trn, loss_val

        scheduler.step()
    model.load_state_dict(best_model_state)
    return model, loss_trn, loss_val